# Graph-spectral covariance

Build a positive-semidefinite covariance matrix from an undirected graph Laplacian. This is a graph-spectral construction, not a generic shortest-path kernel.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd()
while root != root.parent and not (root / 'implementations').exists():
    root = root.parent
sys.path.insert(0, str(root / 'implementations' / 'python' / 'graph-spectral-covariance'))
from graph_covariance import graph_laplacian, graph_spectral_covariance, graph_heat_covariance, covariance_diagnostics


## 1. Define an undirected graph

This path graph is small enough to inspect by hand.

In [ ]:
A = np.array([[0, 1, 0, 0, 0], [1, 0, 1, 0, 0], [0, 1, 0, 1, 0], [0, 0, 1, 0, 1], [0, 0, 0, 1, 0]], dtype=float)
L = graph_laplacian(A)
eigenvalues, eigenvectors = np.linalg.eigh(L)
print('Laplacian:\\n', L)
print('eigenvalues:', eigenvalues)


## 2. Build a spectral covariance

Every spectral variance is non-negative, so the resulting matrix is positive semidefinite by construction.

In [ ]:
K = graph_spectral_covariance(A, [1.0, 0.8, 0.5, 0.25, 0.1])
print(K.round(3))
print(covariance_diagnostics(K))
plt.imshow(K, interpolation='nearest')
plt.colorbar(label='Covariance')
plt.title('Graph-spectral covariance')
plt.xlabel('Vertex')
plt.ylabel('Vertex')
plt.show()


## 3. Explore heat-kernel smoothing

Increasing diffusion time attenuates high-frequency graph modes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, time in zip(axes, [0.0, 0.5, 2.0]):
    Kt = graph_heat_covariance(A, diffusion_time=time)
    ax.imshow(Kt, interpolation='nearest')
    ax.set_title(f't = {time}')
    ax.set_xlabel('Vertex')
    ax.set_ylabel('Vertex')
plt.tight_layout()
plt.show()


## 4. Compare graph topology

The same spectral recipe can be applied to different undirected graphs, but the resulting covariance is tied to each graph's topology and Laplacian.

In [ ]:
star = np.array([[0,1,1,1,1],[1,0,0,0,0],[1,0,0,0,0],[1,0,0,0,0],[1,0,0,0,0]], dtype=float)
cycle = np.roll(np.eye(5), 1, axis=1) + np.roll(np.eye(5), -1, axis=1)
for name, graph in [('path', A), ('star', star), ('cycle', cycle)]:
    Kt = graph_heat_covariance(graph, diffusion_time=0.8)
    print(name, covariance_diagnostics(Kt))


## 5. Interpret with care

This is a finite undirected graph model. It is not a flow-aware network covariance, a shortest-path stationary model or a generalised-network space-time construction.